In [2]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [3]:
# local embedding model import

MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9278.64it/s]


In [4]:
documents = [
    Document(
        page_content="LangChain retrievers return relevant documents.",
        metadata={"topic": "retriever", "level": "beginner"}
    ),
    Document(
        page_content="MultiQueryRetriever improves retrieval using multiple queries.",
        metadata={"topic": "retriever", "level": "advanced"}
    ),
    Document(
        page_content="Chroma is a vector database.",
        metadata={"topic": "vector_store", "level": "beginner"}
    ),
    Document(
        page_content="FAISS is optimized for vector similarity search.",
        metadata={"topic": "vector_store", "level": "advanced"}
    ),
]

vector_store = Chroma(
    collection_name="metadata_filter_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_metadata_filter"
)

vector_store.add_documents(documents)

['96fafa2b-1740-4dde-8389-2de0c002bea0',
 'ad1cb112-cf62-41b9-a04f-441335e02ac2',
 '2788d4c4-14eb-4571-aba8-e02ffb2a62e8',
 'fdd67c6e-83e1-4481-aa53-2efe7510db6d']

In [6]:
# শুধু beginner-level retriever document খুঁজছি
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"level": "beginner"}  # metadata filter: শুধু এই condition match করলে document আসবে
    }
)

query = "How do retrievers work?"

results = retriever.invoke(query)


print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")
    print("Metadata:", doc.metadata)


Query: How do retrievers work?
--------------------------------------------------
1. LangChain retrievers return relevant documents.
Metadata: {'level': 'beginner', 'topic': 'retriever'}
2. Chroma is a vector database.
Metadata: {'level': 'beginner', 'topic': 'vector_store'}
